# Amazon Fine Food Reviews — Sentiment ML Backbone

SageMaker Studio notebook for the review-level binary sentiment backbone. It consumes the Gold `model_input` Parquet dataset, compares three classical models, selects the best model using validation F1, evaluates it once on the untouched test split, and uploads a reusable artifact to S3.

This notebook does **not** perform food-aspect detection. The later hybrid lexicon/LDA/aspect-sentiment work consumes `gold/aspect_sentences/`.

## Leakage-safe methodology

- Gold already assigned deterministic, label-stratified `train`, `validation`, and `test` partitions.
- Only the training partition is downsampled to equal class counts.
- Validation and test retain the observed class imbalance.
- TF-IDF vocabulary and IDF weights are fitted on balanced training text only.
- Logistic Regression, Linear SVM, and Multinomial Naive Bayes are compared on validation data.
- The selected model is evaluated once on test data.
- Ratings 1–2 are negative (`0`), ratings 4–5 are positive (`1`), and 3-star reviews are absent from this binary dataset.

## 1. Install dependencies

Run this cell once per new SageMaker Studio kernel. Restart the kernel if Studio requests it.

In [ ]:
%pip install -q pandas pyarrow scikit-learn joblib matplotlib seaborn boto3

## 2. Imports and configuration

In [ ]:
import json
import platform
import tarfile
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

SEED = 42
BUCKET = "amazon-food-reviews-ml-model"
GOLD_MODEL_PREFIX = "gold/model_input/"
MODEL_PREFIX = "models/sentiment-backbone/"
MAX_TRAIN_ROWS_PER_CLASS = 75_000
MAX_TFIDF_FEATURES = 100_000

np.random.seed(SEED)
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
session = boto3.Session()
region = session.region_name or "eu-north-1"
s3 = session.client("s3", region_name=region)

print(f"Region: {region}")
print(f"Gold input: s3://{BUCKET}/{GOLD_MODEL_PREFIX}")
print(f"Training cap per class: {MAX_TRAIN_ROWS_PER_CLASS:,}")

## 3. Download the partitioned Gold model dataset

In [ ]:
local_gold_dir = Path("data/gold_model_input") / run_id
local_gold_dir.mkdir(parents=True, exist_ok=True)

parquet_keys = []
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BUCKET, Prefix=GOLD_MODEL_PREFIX):
    for item in page.get("Contents", []):
        if item["Key"].endswith(".parquet"):
            parquet_keys.append(item["Key"])

if not parquet_keys:
    raise FileNotFoundError(
        f"No Gold model Parquet files at s3://{BUCKET}/{GOLD_MODEL_PREFIX}"
    )

for key in parquet_keys:
    relative_path = Path(key[len(GOLD_MODEL_PREFIX):])
    destination = local_gold_dir / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(BUCKET, key, str(destination))

print(f"Downloaded {len(parquet_keys)} Parquet files to {local_gold_dir}")

## 4. Load and validate train/validation/test partitions

Spark stores `dataset_split` in Hive-style folder names, so the loader reconstructs it from paths such as `dataset_split=train/part-....parquet`.

In [ ]:
def split_from_path(path):
    for part in path.parts:
        if part.startswith("dataset_split="):
            return part.split("=", 1)[1]
    raise ValueError(f"Cannot determine dataset split from {path}")


frames = []
for parquet_file in sorted(local_gold_dir.rglob("*.parquet")):
    frame = pd.read_parquet(parquet_file)
    frame["dataset_split"] = split_from_path(parquet_file)
    frames.append(frame)

data = pd.concat(frames, ignore_index=True)
required_columns = {"record_id", "clean_text", "label", "dataset_split"}
missing_columns = sorted(required_columns - set(data.columns))
if missing_columns:
    raise ValueError(f"Gold model data is missing columns: {missing_columns}")

data = data.dropna(subset=["record_id", "clean_text", "label"]).copy()
data["clean_text"] = data["clean_text"].astype(str).str.strip()
data["label"] = pd.to_numeric(data["label"], errors="coerce")
data = data.dropna(subset=["label"])
data["label"] = data["label"].astype(int)
data = data[
    data["clean_text"].ne("")
    & data["label"].isin([0, 1])
    & data["dataset_split"].isin(["train", "validation", "test"])
].reset_index(drop=True)

if data.empty or data["label"].nunique() != 2:
    raise ValueError("Gold model input must contain non-empty text and both labels")
if data["record_id"].duplicated().any():
    raise ValueError("Duplicate record_id values found in Gold model input")
if data["clean_text"].duplicated().any():
    raise ValueError("Duplicate clean_text values found; leakage protection failed")

split_sets = {
    name: set(data.loc[data["dataset_split"] == name, "record_id"])
    for name in ["train", "validation", "test"]
}
if split_sets["train"] & split_sets["validation"]:
    raise ValueError("Train and validation IDs overlap")
if split_sets["train"] & split_sets["test"]:
    raise ValueError("Train and test IDs overlap")
if split_sets["validation"] & split_sets["test"]:
    raise ValueError("Validation and test IDs overlap")

distribution = pd.crosstab(data["dataset_split"], data["label"], margins=True)
display(distribution)
print(f"Validated Gold rows: {len(data):,}")

In [ ]:
train_full = data[data["dataset_split"] == "train"].copy()
validation = data[data["dataset_split"] == "validation"].copy()
test = data[data["dataset_split"] == "test"].copy()

train_counts = train_full["label"].value_counts()
rows_per_class = min(int(train_counts.min()), MAX_TRAIN_ROWS_PER_CLASS)
train_balanced = (
    pd.concat(
        [
            group.sample(n=rows_per_class, random_state=SEED)
            for _, group in train_full.groupby("label", sort=True)
        ],
        ignore_index=True,
    )
    .sample(frac=1.0, random_state=SEED)
    .reset_index(drop=True)
)

print(f"Full training rows: {len(train_full):,}")
print(f"Balanced training rows: {len(train_balanced):,}")
print(f"Validation rows (unchanged): {len(validation):,}")
print(f"Test rows (unchanged): {len(test):,}")
display(
    pd.DataFrame(
        {
            "train_full": train_full["label"].value_counts().sort_index(),
            "train_balanced": train_balanced["label"].value_counts().sort_index(),
            "validation": validation["label"].value_counts().sort_index(),
            "test": test["label"].value_counts().sort_index(),
        }
    ).fillna(0).astype(int)
)

## 5. Fit TF-IDF on balanced training text only

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.98,
    max_features=MAX_TFIDF_FEATURES,
    sublinear_tf=True,
    dtype=np.float32,
)

feature_start = time.perf_counter()
X_train = vectorizer.fit_transform(train_balanced["clean_text"])
X_validation = vectorizer.transform(validation["clean_text"])
X_test = vectorizer.transform(test["clean_text"])
y_train = train_balanced["label"].to_numpy()
y_validation = validation["label"].to_numpy()
y_test = test["label"].to_numpy()

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Training matrix: {X_train.shape}, nonzero={X_train.nnz:,}")
print(f"Feature preparation: {time.perf_counter() - feature_start:.1f}s")

## 6. Train and compare classical models on validation data

In [ ]:
models = {
    "logistic_regression": LogisticRegression(
        C=1.0, solver="liblinear", max_iter=500, random_state=SEED
    ),
    "linear_svm": LinearSVC(C=1.0, random_state=SEED),
    "multinomial_nb": MultinomialNB(alpha=1.0),
}


def model_scores(model, features):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(features)
    return model.predict(features)


def evaluate_predictions(y_true, predictions, scores):
    return {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
    }


validation_results = []
fitted_models = {}
for model_name, candidate in models.items():
    started = time.perf_counter()
    candidate.fit(X_train, y_train)
    predictions = candidate.predict(X_validation)
    scores = model_scores(candidate, X_validation)
    metrics = evaluate_predictions(y_validation, predictions, scores)
    metrics.update(
        {
            "model": model_name,
            "fit_seconds": round(time.perf_counter() - started, 2),
        }
    )
    validation_results.append(metrics)
    fitted_models[model_name] = candidate

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values(["f1", "average_precision"], ascending=False)
    .reset_index(drop=True)
)
display(validation_results_df)

In [ ]:
plot_metrics = validation_results_df.set_index("model")[
    ["accuracy", "balanced_accuracy", "precision", "recall", "f1"]
]
ax = plot_metrics.plot(kind="bar", figsize=(11, 5), ylim=(0, 1))
ax.set_title("Validation performance by model")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.legend(loc="lower right", ncol=3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Select on validation and evaluate once on untouched test data

In [ ]:
best_model_name = validation_results_df.iloc[0]["model"]
best_classifier = fitted_models[best_model_name]
test_predictions = best_classifier.predict(X_test)
test_scores = model_scores(best_classifier, X_test)
test_metrics = evaluate_predictions(y_test, test_predictions, test_scores)
test_metrics["confusion_matrix"] = confusion_matrix(y_test, test_predictions).tolist()
test_metrics["classification_report"] = classification_report(
    y_test,
    test_predictions,
    target_names=["negative", "positive"],
    output_dict=True,
    zero_division=0,
)

print(f"Selected model: {best_model_name}")
display(pd.DataFrame([test_metrics]).drop(columns=["confusion_matrix", "classification_report"]))
print(classification_report(
    y_test, test_predictions, target_names=["negative", "positive"], zero_division=0
))

matrix = np.asarray(test_metrics["confusion_matrix"])
sns.heatmap(
    matrix, annot=True, fmt=",d", cmap="Blues",
    xticklabels=["negative", "positive"],
    yticklabels=["negative", "positive"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Test confusion matrix — {best_model_name}")
plt.tight_layout()
plt.show()

## 8. Inspect errors

This qualitative check is useful for finding negation, mixed sentiment, rating noise, and food-domain phrases that motivate the later aspect layer.

In [ ]:
error_analysis = test[["record_id", "clean_text", "label", "score"]].copy()
error_analysis["prediction"] = test_predictions
error_analysis["model_score"] = test_scores

false_positives = error_analysis[
    (error_analysis["label"] == 0) & (error_analysis["prediction"] == 1)
].sort_values("model_score", ascending=False)
false_negatives = error_analysis[
    (error_analysis["label"] == 1) & (error_analysis["prediction"] == 0)
].sort_values("model_score", ascending=True)

print("Most confident false positives:")
display(false_positives.head(10))
print("Most confident false negatives:")
display(false_negatives.head(10))

## 9. Package and upload the selected pipeline

The artifact contains the fitted TF-IDF vectorizer and classifier together, so inference applies exactly the same feature transformation used during training.

In [ ]:
selected_pipeline = Pipeline(
    [("tfidf", vectorizer), ("classifier", best_classifier)]
)

artifact_dir = Path("artifacts") / run_id
artifact_dir.mkdir(parents=True, exist_ok=True)
model_file = artifact_dir / "model.joblib"
metrics_file = artifact_dir / "metrics.json"
archive_file = artifact_dir / "model.tar.gz"

metadata = {
    "run_id": run_id,
    "selected_model": best_model_name,
    "selection_metric": "validation_f1",
    "label_mapping": {"0": "negative (ratings 1-2)", "1": "positive (ratings 4-5)"},
    "gold_input": f"s3://{BUCKET}/{GOLD_MODEL_PREFIX}",
    "training_rows_full": int(len(train_full)),
    "training_rows_balanced": int(len(train_balanced)),
    "validation_rows": int(len(validation)),
    "test_rows": int(len(test)),
    "tfidf_vocabulary_size": int(len(vectorizer.vocabulary_)),
    "validation_results": validation_results_df.to_dict(orient="records"),
    "test_metrics": test_metrics,
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
}

joblib.dump(selected_pipeline, model_file)
with metrics_file.open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

with tarfile.open(archive_file, "w:gz") as archive:
    archive.add(model_file, arcname="model.joblib")
    archive.add(metrics_file, arcname="metrics.json")

artifact_key = f"{MODEL_PREFIX}{run_id}/model.tar.gz"
metrics_key = f"{MODEL_PREFIX}{run_id}/metrics.json"
s3.upload_file(str(archive_file), BUCKET, artifact_key)
s3.upload_file(str(metrics_file), BUCKET, metrics_key)

print(f"Local model: {model_file}")
print(f"Model artifact: s3://{BUCKET}/{artifact_key}")
print(f"Metrics: s3://{BUCKET}/{metrics_key}")

## 10. Test food-review predictions

In [ ]:
sample_reviews = [
    "The coffee tastes rich and fresh, and I would definitely buy it again.",
    "The package arrived crushed and the food was stale and inedible.",
    "The flavor is excellent, but the shipping was late and the box was damaged.",
]
sample_predictions = selected_pipeline.predict(sample_reviews).astype(int)
sample_output = pd.DataFrame(
    {
        "review": sample_reviews,
        "predicted_label": sample_predictions,
        "predicted_sentiment": np.where(sample_predictions == 1, "positive", "negative"),
    }
)
if hasattr(best_classifier, "predict_proba"):
    sample_output["positive_probability"] = selected_pipeline.predict_proba(sample_reviews)[:, 1]
else:
    sample_output["decision_score"] = selected_pipeline.decision_function(sample_reviews)
display(sample_output)

## Next modeling stages

This establishes the scalable classical ML backbone. A Bi-LSTM comparison should be implemented separately using the same Gold splits and a tokenizer fitted only on training data. The food-domain contribution then uses `gold/aspect_sentences/` for seeded aspect lexicons, LDA validation, sentence-level aspect sentiment, and actionable product/time/helpfulness insights.